# 策略搜索

In [1]:
# 导入策略插件函数等必须的包
import os
import pandas as pd
from shaphypetune.policy.rules_finder import *
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth',1000)

# 准备需要做策略的数据样本
data = pd.read_csv('rules_finder_sample_data.csv')
# data = pd.read_csv('./rules_finder_sample_data.csv')
data = data[(data.if_t1==1)&(data.shouxin_date>='2020-08-01')]

# 准备一个列表，包含数据中可以用来做策略的模型分的列名
score_list = ['score_chal','score_add','score_orig','score_bc1','score_bc2','v3_newfeat30_score']
data['shouxin_month'] = data.shouxin_date.apply(lambda x:x.replace("-","")).str[:6]
data['shouxin_month'] = data['shouxin_month'].astype(np.int64)

/home/ubuntu/miniforge3/envs/basefrm_latest/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ubuntu/miniforge3/envs/basefrm_latest/lib/python3.11/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## gbl策略搜索工具 

### 模型池内搜索全局最优策略，建议变量是模型分（分数和风险正相关）

In [2]:
# 调用策略工具，需要两个参数来接收函数返回值：
#                        第一个是dataframe，包含各策略内容及其效果；第二个是字典，包含各单模型分的各卡点及其效果。
df_rule,detail = gbl_strategy_finder(
                                 data = data[data.shouxin_date<'2020-10-01'], # 需要做策略的样本
                                 label='fpd1',                                # 样本中y的列名
                                 params=score_list,                           # 列表，包含用于做策略的模型分列名
                                 bad_lift=2,                                  # 单模型分拒绝样本的最低坏率倍数容忍值
                                 good_lift=0.4,                               # 单模型分通过样本的最高坏率倍数容忍值
                                 mode='ar',                                   # 单模型分选取卡点的模式，有ar、risk、effic三种模式，推荐使用ar模式
                                 method='gain',                               # 多个模型分结合的方法，有union、gain两种方法，推荐使用gain方法
                                 n_rounds=15,                                 # method=union时，只需传1即可；method=gain时，推荐传10以上
                                 pass_limit=0,                                # 通过样本对总样本的占比，若发现整体策略在oot上效果同做策略的样本上有明显差异，可以调大此值以防止过拟合，建议从0.01开始尝试，过大可能会导致策略效果明显下降。
                                 reject_limit=0,                              # 拒绝样本对总样本的占比，同上。
                                 random_state=0                               # 随机数种子
                                )

----------------------------------------   round1   complete----------------------------------------
----------------------------------------   round2   complete----------------------------------------
----------------------------------------   round3   complete----------------------------------------
----------------------------------------   round4   complete----------------------------------------
----------------------------------------   round5   complete----------------------------------------
----------------------------------------   round6   complete----------------------------------------
----------------------------------------   round7   complete----------------------------------------
----------------------------------------   round8   complete----------------------------------------
----------------------------------------   round9   complete----------------------------------------
---------------------------------------   round10   complete-------------------------------

In [3]:
df_rule

,rule_str,risk_cnt,risk,lift,pass_cnt,AR
0,score_orig<=-0.175,7.0,0.026,0.271,274,0.1
1,v3_newfeat30_score<=0.026 and not(v3_newfeat30_score>0.026 or score_chal>0.39),29.0,0.035,0.365,817,0.299
2,score_add<=0.011 and not(score_bc2>0.037),11.0,0.037,0.386,299,0.109
3,score_add<=0.011 and not(score_bc2>0.037),11.0,0.037,0.386,299,0.109
4,v3_newfeat30_score<=0.026 and not(score_chal>0.39 or v3_newfeat30_score>0.026),29.0,0.035,0.365,817,0.299
5,v3_newfeat30_score<=0.026 and not(v3_newfeat30_score>0.026 or score_chal>0.39),29.0,0.035,0.365,817,0.299
6,score_orig<=-0.175,7.0,0.026,0.271,274,0.1
7,score_add<=0.011 and not(score_bc2>0.037),11.0,0.037,0.386,299,0.109
8,score_add<=0.011 and not(score_bc2>0.037),11.0,0.037,0.386,299,0.109
9,v3_newfeat30_score<=0.026 and not(score_chal>0.39 or v3_newfeat30_score>0.026),29.0,0.035,0.365,817,0.299


In [4]:
df_reports = atsr_report_clf(data[data["shouxin_date"]<'2020-10-01'],  #数据集
                             df_rule,                                  #生成的规则集
                             rule_col='rule_str',                      #规则集变量名
                             label='fpd1',                             #目标变量
                             groups='shouxin_month',                   #指定切片变量 
                             id_col='cust_no')
df_reports

             Processing score_add<=0.011 and not(score_bc2>0.037)              : 100%|██████████| 15/15 [00:00<00:00, 36.69it/s]                                     


cnt  \
shouxin_month                                                                  202008   
score_orig<=-0.175                                                              135.0   
v3_newfeat30_score<=0.026 and not(v3_newfeat30_score>0.026 or score_chal>0.39)  456.0   
score_add<=0.011 and not(score_bc2>0.037)                                       151.0   
score_add<=0.011 and not(score_bc2>0.037)                                       151.0   
v3_newfeat30_score<=0.026 and not(score_chal>0.39 or v3_newfeat30_score>0.026)  456.0   
v3_newfeat30_score<=0.026 and not(v3_newfeat30_score>0.026 or score_chal>0.39)  456.0   
score_orig<=-0.175                                                              135.0   
score_add<=0.011 and not(score_bc2>0.037)                                       151.0   
score_add<=0.011 and not(score_bc2>0.037)                                       151.0   
v3_newfeat30_score<=0.026 and not(score_chal>0.39 or v3_newfeat30_score>0.026)  456.0   
score_add<=0.011 and not(score_bc2>0.037)                                       151.0   
score_orig<=-0.175                                                              135.0   
score_orig<=-0.175                                                              135.0   
v3_newfeat30_score<=0.026 and not(v3_newfeat30_score>0.026 or score_chal>0.39)  456.0   
score_add<=0.011 and not(score_bc2>0.037)                                       151.0   

                                                                                       \
shouxin_month                                                                  202009   
score_orig<=-0.175                                                              139.0   
v3_newfeat30_score<=0.026 and not(v3_newfeat30_score>0.026 or score_chal>0.39)  361.0   
score_add<=0.011 and not(score_bc2>0.037)                                       148.0   
score_add<=0.011 and not(score_bc2>0.037)                                       148.0   
v3_newfeat30_score<=0.026 and not(score_chal>0.39 or v3_newfeat30_score>0.026)  361.0   
v3_newfeat30_score<=0.026 and not(v3_newfeat30_score>0.026 or score_chal>0.39)  361.0   
score_orig<=-0.175                                                              139.0   
score_add<=0.011 and not(score_bc2>0.037)                                       148.0   
score_add<=0.011 and not(score_bc2>0.037)                                       148.0   
v3_newfeat30_score<=0.026 and not(score_chal>0.39 or v3_newfeat30_score>0.026)  361.0   
score_add<=0.011 and not(score_bc2>0.037)                                       148.0   
score_orig<=-0.175                                                              139.0   
score_orig<=-0.175                                                              139.0   
v3_newfeat30_score<=0.026 and not(v3_newfeat30_score>0.026 or score_chal>0.39)  361.0   
score_add<=0.011 and not(score_bc2>0.037)                                       148.0   

                                                                                       \
shouxin_month                                                                     All   
score_orig<=-0.175                                                              274.0   
v3_newfeat30_score<=0.026 and not(v3_newfeat30_score>0.026 or score_chal>0.39)  817.0   
score_add<=0.011 and not(score_bc2>0.037)                                       299.0   
score_add<=0.011 and not(score_bc2>0.037)                                       299.0   
v3_newfeat30_score<=0.026 and not(score_chal>0.39 or v3_newfeat30_score>0.026)  817.0   
v3_newfeat30_score<=0.026 and not(v3_newfeat30_score>0.026 or score_chal>0.39)  817.0   
score_orig<=-0.175                                                              274.0   
score_add<=0.011 and not(score_bc2>0.037)                                       299.0   
score_add<=0.011 and not(score_bc2>0.037)                                       299.0   
v3_newfeat30_score<=0.026 and not(score_chal>0.39 or v3_newfeat30_score>0.0

In [ ]:
df_reports = atsr_report_clf(data[data["shouxin_date"]<'2020-10-01'],  #数据集
                             df_rule,                                  #生成的规则集
                             rule_col='rule_str',                      #规则集变量名
                             label='fpd1',                             #目标变量
                             groups='shouxin_month',                   #指定切片变量 
                             id_col='cust_no')
df_reports

             Processing score_add<=0.011 and not(score_bc2>0.037)              : 100%|██████████| 15/15 [00:00<00:00, 67.94it/s]                                    


cnt  \
shouxin_month                                                                  202008   
score_orig<=-0.175                                                              135.0   
v3_newfeat30_score<=0.026 and not(v3_newfeat30_score>0.026 or score_chal>0.39)  456.0   
score_add<=0.011 and not(score_bc2>0.037)                                       151.0   
score_add<=0.011 and not(score_bc2>0.037)                                       151.0   
v3_newfeat30_score<=0.026 and not(score_chal>0.39 or v3_newfeat30_score>0.026)  456.0   
v3_newfeat30_score<=0.026 and not(v3_newfeat30_score>0.026 or score_chal>0.39)  456.0   
score_orig<=-0.175                                                              135.0   
score_add<=0.011 and not(score_bc2>0.037)                                       151.0   
score_add<=0.011 and not(score_bc2>0.037)                                       151.0   
v3_newfeat30_score<=0.026 and not(score_chal>0.39 or v3_newfeat30_score>0.026)  456.0   
score_add<=0.011 and not(score_bc2>0.037)                                       151.0   
score_orig<=-0.175                                                              135.0   
score_orig<=-0.175                                                              135.0   
v3_newfeat30_score<=0.026 and not(v3_newfeat30_score>0.026 or score_chal>0.39)  456.0   
score_add<=0.011 and not(score_bc2>0.037)                                       151.0   

                                                                                       \
shouxin_month                                                                  202009   
score_orig<=-0.175                                                              139.0   
v3_newfeat30_score<=0.026 and not(v3_newfeat30_score>0.026 or score_chal>0.39)  361.0   
score_add<=0.011 and not(score_bc2>0.037)                                       148.0   
score_add<=0.011 and not(score_bc2>0.037)                                       148.0   
v3_newfeat30_score<=0.026 and not(score_chal>0.39 or v3_newfeat30_score>0.026)  361.0   
v3_newfeat30_score<=0.026 and not(v3_newfeat30_score>0.026 or score_chal>0.39)  361.0   
score_orig<=-0.175                                                              139.0   
score_add<=0.011 and not(score_bc2>0.037)                                       148.0   
score_add<=0.011 and not(score_bc2>0.037)                                       148.0   
v3_newfeat30_score<=0.026 and not(score_chal>0.39 or v3_newfeat30_score>0.026)  361.0   
score_add<=0.011 and not(score_bc2>0.037)                                       148.0   
score_orig<=-0.175                                                              139.0   
score_orig<=-0.175                                                              139.0   
v3_newfeat30_score<=0.026 and not(v3_newfeat30_score>0.026 or score_chal>0.39)  361.0   
score_add<=0.011 and not(score_bc2>0.037)                                       148.0   

                                                                                       \
shouxin_month                                                                     All   
score_orig<=-0.175                                                              274.0   
v3_newfeat30_score<=0.026 and not(v3_newfeat30_score>0.026 or score_chal>0.39)  817.0   
score_add<=0.011 and not(score_bc2>0.037)                                       299.0   
score_add<=0.011 and not(score_bc2>0.037)                                       299.0   
v3_newfeat30_score<=0.026 and not(score_chal>0.39 or v3_newfeat30_score>0.026)  817.0   
v3_newfeat30_score<=0.026 and not(v3_newfeat30_score>0.026 or score_chal>0.39)  817.0   
score_orig<=-0.175                                                              274.0   
score_add<=0.011 and not(score_bc2>0.037)                                       299.0   
score_add<=0.011 and not(score_bc2>0.037)                                       299.0   
v3_newfeat30_score<=0.026 and not(score_chal>0.39 or v3_newfeat30_score>0.0

### gbl搜索工具特点

gbl策略工具对所有模型分进行**全局搜索**，并棍据指定的目标，选出最佳规则组合形成策略，**策略搜索精度高，如果对策略精度要求高推荐使用此工具**，但耗时较长（demo条件下迭代15轮耗时45秒）。

## frdm策略快搜工具 

### 模型池内快速搜索有效策略，建议变量是模型分（分数和风险正相关）

In [3]:
rtn, _, _, _ =frdm_strategy_finder(data = data[data["shouxin_date"]<'2020-10-01'], # 分析样本：数据必须有日期变量和Y变量
                                 groups = 'shouxin_month',          # 日期变量名，日期变量约定格式为 202001，分月查看策略效果
                                 id_col = "cust_no",                # 主键
                                 label='fpd1',                      # y变量名
                                 params=score_list,                 # 入模变量名列表
                                 n_cut=10,                          # 变量分箱数
                                 n_rj_cut=3,                        # 拒绝组模型分切点随机抽取的范围，该数值不可超过变量分箱数，即n_cut，e.g 如取3，则将在模型分后2个切点中随机生成策略
                                 n_pass_cut=4,                      # 通过组模型分切点随机抽取的范围，该数值不可超过变量分箱数，即n_cut，e.g 如取4，则将在模型分前3个切点中随机生成策略
                                 use_tg = [],                       # 用于通过规则的变量
                                 use_rj = [],                       # 用于拒绝规则的变量
                                 bad_lift=2,                        # 模型分箱后最后一组风险倍数的阈值
                                 good_lift=0.4,                     # 模型分箱后第一组风险倍数阈值
                                 n_rounds=20,                       # 策略随机次数
                                 threds=[0.05,0.10],                # 策略需要满足的条件[风险上限，通过率下限]
                                 missing_val_ls=[-99],              # 缺失值认定列表，缺失值将会设置为9999
                                 random_state = 0                   # 随机数种子
                                )

-----开始检查数据------
groups检查正确
id_col检查正确
label检查正确
-----数据检查完毕------
第4条满足条件,风险为0.04461,通过率为0.39371
策略为( score_add<=0.013853 or score_orig<=-0.175131 or v3_newfeat30_score<=0.025993 ) and not ( v3_newfeat30_score>=0.067697 ) 
分月通过率和风险如下：
                fpd  tg_cnt  tg_cnt2   cnt  tg_rati     fpd%
shouxin_month                                               
202008         27.0     592      592  1509  0.39231  0.04561
202009         21.0     484      484  1224  0.39542  0.04339
------------------------
第12条满足条件,风险为0.04786,通过率为0.35931
策略为( score_add<=0.013853 or score_orig<=0.09346 or v3_newfeat30_score<=0.021608 ) and not ( v3_newfeat30_score>=0.067697 ) 
分月通过率和风险如下：
                fpd  tg_cnt  tg_cnt2   cnt  tg_rati     fpd%
shouxin_month                                               
202008         28.0     532      532  1509  0.35255  0.05263
202009         19.0     450      450  1224  0.36765  0.04222
------------------------
第15条满足条件,风险为0.05,通过率为0.43908
策略为( score_add<=0.017251 or 

In [6]:
# 查看返回的各策略及其效果

df_reports2 = atsr_report_clf(data[data["shouxin_date"]<'2020-10-01'], #数据集
                             rtn,                                      #生成的规则集
                             rule_col='rule_str',                      #规则集变量名
                             label='fpd1',                             #目标变量
                             groups='shouxin_month',                   #指定切片变量 
                             id_col='cust_no')
df_reports2

             Processing ( score_add<=0.017251 or score_orig<=-0.175131 or v3_newfeat30_score<=0.025993 ) and not ( v3_newfeat30_score>=0.067697 )               : 100%|██████████| 3/3 [00:00<00:00, 64.83it/s]


cnt  \
shouxin_month                                                                                                              202008   
( score_add<=0.013853 or score_orig<=-0.175131 or v3_newfeat30_score<=0.025993 ) and not ( v3_newfeat30_score>=0.067697 )   592.0   
( score_add<=0.013853 or score_orig<=0.09346 or v3_newfeat30_score<=0.021608 ) and not ( v3_newfeat30_score>=0.067697 )     532.0   
( score_add<=0.017251 or score_orig<=-0.175131 or v3_newfeat30_score<=0.025993 ) and not ( v3_newfeat30_score>=0.067697 )   654.0   

                                                                                                                                   \
shouxin_month                                                                                                              202009   
( score_add<=0.013853 or score_orig<=-0.175131 or v3_newfeat30_score<=0.025993 ) and not ( v3_newfeat30_score>=0.067697 )   484.0   
( score_add<=0.013853 or score_orig<=0.09346 or v3_newfeat30_score<=0.021608 ) and not ( v3_newfeat30_score>=0.067697 )     450.0   
( score_add<=0.017251 or score_orig<=-0.175131 or v3_newfeat30_score<=0.025993 ) and not ( v3_newfeat30_score>=0.067697 )   546.0   

                                                                                                                                    \
shouxin_month                                                                                                                  All   
( score_add<=0.013853 or score_orig<=-0.175131 or v3_newfeat30_score<=0.025993 ) and not ( v3_newfeat30_score>=0.067697 )   1076.0   
( score_add<=0.013853 or score_orig<=0.09346 or v3_newfeat30_score<=0.021608 ) and not ( v3_newfeat30_score>=0.067697 )      982.0   
( score_add<=0.017251 or score_orig<=-0.175131 or v3_newfeat30_score<=0.025993 ) and not ( v3_newfeat30_score>=0.067697 )   1200.0   

                                                                                                                             fpd1  \
shouxin_month                                                                                                              202008   
( score_add<=0.013853 or score_orig<=-0.175131 or v3_newfeat30_score<=0.025993 ) and not ( v3_newfeat30_score>=0.067697 )    27.0   
( score_add<=0.013853 or score_orig<=0.09346 or v3_newfeat30_score<=0.021608 ) and not ( v3_newfeat30_score>=0.067697 )      28.0   
( score_add<=0.017251 or score_orig<=-0.175131 or v3_newfeat30_score<=0.025993 ) and not ( v3_newfeat30_score>=0.067697 )    34.0   

                                                                                                                                   \
shouxin_month                                                                                                              202009   
( score_add<=0.013853 or score_orig<=-0.175131 or v3_newfeat30_score<=0.025993 ) and not ( v3_newfeat30_score>=0.067697 )    21.0   
( score_add<=0.013853 or score_orig<=0.09346 or v3_newfeat30_score<=0.021608 ) and not ( v3_newfeat30_score>=0.067697 )      19.0   
( score_add<=0.017251 or score_orig<=-0.175131 or v3_newfeat30_score<=0.025993 ) and not ( v3_newfeat30_score>=0.067697 )    26.0   

                                                                                                                                  \
shouxin_month                                                                                                                All   
( score_add<=0.013853 or score_orig<=-0.175131 or v3_newfeat30_score<=0.025993 ) and not ( v3_newfeat30_score>=0.067697 )   48.0   
( score_add<=0.013853 or score_orig<=0.09346 or v3_newfeat30_score<=0.021608 ) and not ( v3_newfeat30_score>=0.067697 )     47.0   
( score_add<=0.017251 or score_orig<=-0.175131 or v3_newfeat30_score<=0.025993 ) and not ( v3_newfeat30_score>=0.067697 )   60.0   

                                                                                                                            fpd1_p

### frdm搜索工具特点

1、frdm策略工具根据模型分的区分表现，随机选取用户指定范围内的一个模型分卡点来作为通过规则或拒绝规则，将所有通过规则和拒绝规则合并作为最终的策略输出，同时也可控制n_rounds进行多轮次迭代,保留满足要求的策略结果；

2、相比gbl策略工具的全局搜索，frdm策略工具**缩小搜索范围并进行随机抽样**，牺牲了一定精度但提高了搜索效率（demo条件下迭代15轮次只需约2s），**如果模型分数量非常多且对速度有一定的要求可以考虑使用此工具**。

## atsr规则搜索工具

### 针对多变量的规则搜索工具，特点是考虑维度广阔，可以输出决策过程中的全决策路径规则,同时附带高频出现在规则中的变量清单和策略分支

In [8]:
#自动生成规则  - 二分类
df_class = atsr_rule_finder_clf(data = data[data["shouxin_date"]<'2020-10-01'],    #数据集
                                params=score_list,                                 #规则生成所使用的变量空间
                                label='fpd4',                                      #指定目标变量
                                n_rounds = 20,                                     #算法执行轮数
                                num_feature_min =2 ,                               #规则包含的最小变量数
                                num_feature_max = 5,                               #规则包含的最大变量数
                                depth=3,                                           #决策树深度，与生成的规则复杂度有关
                                min_samples_leaf=100)                              #叶节点最小样本数

100%|██████████| 20/20 [00:00<00:00, 151.70it/s]


In [9]:
df_class.head()

,rule_str,node0,node1,bad_rate
0,(v3_newfeat30_score>0.058650387451052666) & (v3_newfeat30_score<=0.1321147307753563) & (score_add>0.06231122277677059),0.900000,0.100000,0.100000
1,(v3_newfeat30_score>0.1321147307753563),0.913669,0.086331,0.086331
2,(score_add>0.06231122277677059) & (v3_newfeat30_score<=0.1321147307753563),0.932367,0.067633,0.067633
3,(v3_newfeat30_score<=0.058650387451052666) & (score_add>0.06231122277677059),0.962617,0.037383,0.037383
4,(v3_newfeat30_score<=0.1321147307753563),0.981110,0.018890,0.018890


In [10]:
#生成规则报告 - 分类
df_reports = atsr_report_clf(data = data[data["shouxin_date"]<'2020-10-01'],   #数据集
                             rule_df = df_class,                               #规则集
                             rule_col='rule_str',                              #规则集变量
                             label='fpd1',                                     #目标变量
                             groups='shouxin_month',                           #指定切片变量
                             id_col='cust_no'
                            )                             
df_reports.head()

             Processing (score_bc2<=0.019347026012837887) & (v3_newfeat30_score<=0.13183362782001495) & (score_orig<=4.869394540786743)              : 100%|██████████| 82/82 [00:01<00:00, 69.44it/s]       


cnt  \
shouxin_month                                                                                                           202008   
(v3_newfeat30_score>0.058650387451052666) & (v3_newfeat30_score<=0.1321147307753563) & (score_add>0.06231122277677059)    58.0   
(v3_newfeat30_score>0.1321147307753563)                                                                                   85.0   
(score_add>0.06231122277677059) & (v3_newfeat30_score<=0.1321147307753563)                                               123.0   
(v3_newfeat30_score<=0.058650387451052666) & (score_add>0.06231122277677059)                                              65.0   
(v3_newfeat30_score<=0.1321147307753563)                                                                                1424.0   

                                                                                                                                \
shouxin_month                                                                                                           202009   
(v3_newfeat30_score>0.058650387451052666) & (v3_newfeat30_score<=0.1321147307753563) & (score_add>0.06231122277677059)    42.0   
(v3_newfeat30_score>0.1321147307753563)                                                                                   54.0   
(score_add>0.06231122277677059) & (v3_newfeat30_score<=0.1321147307753563)                                                84.0   
(v3_newfeat30_score<=0.058650387451052666) & (score_add>0.06231122277677059)                                              42.0   
(v3_newfeat30_score<=0.1321147307753563)                                                                                1170.0   

                                                                                                                                \
shouxin_month                                                                                                              All   
(v3_newfeat30_score>0.058650387451052666) & (v3_newfeat30_score<=0.1321147307753563) & (score_add>0.06231122277677059)   100.0   
(v3_newfeat30_score>0.1321147307753563)                                                                                  139.0   
(score_add>0.06231122277677059) & (v3_newfeat30_score<=0.1321147307753563)                                               207.0   
(v3_newfeat30_score<=0.058650387451052666) & (score_add>0.06231122277677059)                                             107.0   
(v3_newfeat30_score<=0.1321147307753563)                                                                                2594.0   

                                                                                                                         fpd1  \
shouxin_month                                                                                                          202008   
(v3_newfeat30_score>0.058650387451052666) & (v3_newfeat30_score<=0.1321147307753563) & (score_add>0.06231122277677059)   13.0   
(v3_newfeat30_score>0.1321147307753563)                                                                                  16.0   
(score_add>0.06231122277677059) & (v3_newfeat30_score<=0.1321147307753563)                                               20.0   
(v3_newfeat30_score<=0.058650387451052666) & (score_add>0.06231122277677059)                                              7.0   
(v3_newfeat30_score<=0.1321147307753563)                                                                                127.0   

                                                                                                                               \
shouxin_month                                                                                                          202009   
(v3_newfeat30_score>0.058650387451052666) & (v3_newfeat30_score<=0.1321147307753563) & (score_add>0.06231122277677059)    8.0   
(v3_newfeat30_score>0.1321147307753563)                                                            

In [11]:
# 生成特征重要性和高频区间
varImp = atsrRuleVarImp(df_class, rule_col='rule_str')
varImp[0].style

,var_name,imp
0,score_add,32
1,score_bc1,26
2,score_bc2,28
3,score_chal,26
4,score_orig,28
5,v3_newfeat30_score,26


In [12]:
varImp[1].style

,var_name,dir,threshold,rule
0,score_add,<=,0.062311,(score_add<=0.06231122277677059)
1,score_bc1,<=,0.055480,(score_bc1<=0.05547995679080486)
2,score_bc2,<=,0.034869,(score_bc2<=0.03486856445670128)
3,score_chal,<=,0.366243,(score_chal<=0.36624298989772797)
4,score_orig,<=,4.869395,(score_orig<=4.869394540786743)
5,v3_newfeat30_score,<=,0.132115,(v3_newfeat30_score<=0.1321147307753563)
6,score_add,>,0.016949,(score_add>0.016948901116847992)
7,score_bc1,>,0.047167,(score_bc1>0.04716711491346359)
8,score_bc2,>,0.022476,(score_bc2>0.022475991863757372)
9,score_chal,>,0.272628,(score_chal>0.2726278305053711)


In [14]:
#自动生成规则 - 回归
df_class_reg = atsr_rule_finder_reg(data = data[data["shouxin_date"]<'2020-10-01'],     #数据集
                                n_rounds = 50,                                          #算法执行轮数
                                num_feature_min =2 ,                                    #规则包含的最小变量数 
                                num_feature_max = 5,                                    #规则包含的最大变量数 
                                params=score_list,                                      #规则生成所使用的变量空间
                                label='fpd1',                                           #目标变量
                                depth=3,                                                #指定切片变量
                                min_samples_leaf=100)                                   #叶节点最小样本数

100%|██████████| 50/50 [00:00<00:00, 174.36it/s]


In [15]:
#生成规则报告
df_reports_reg = atsr_report_reg(data = data[data["shouxin_date"]<'2020-10-01'], #数据集
                             rule_df = df_class_reg,                             #规则集
                             rule_col='rule_str',                                #规则集变量
                             label='fpd1',                                       #指定目标变量（连续变量）
                             groups='shouxin_month',                             #指定切片变量
                             id_col='cust_no'
                             )                                    
df_reports_reg.head()

             Processing (score_bc1>0.018574872985482216) & (score_orig>-0.16815992444753647) & (score_orig<=1.9360529780387878)              : 100%|██████████| 102/102 [00:01<00:00, 73.23it/s]                 


cnt  \
shouxin_month                                                                                                              202008   
(score_orig<=-0.15712326765060425) & (v3_newfeat30_score<=0.02879201900213957)                                              100.0   
(score_bc1<=0.019156343303620815) & (v3_newfeat30_score>0.02879201900213957) & (v3_newfeat30_score<=0.048337794840335846)    62.0   
(v3_newfeat30_score<=0.02879201900213957)                                                                                   530.0   
(score_orig>-0.15712326765060425) & (v3_newfeat30_score<=0.02879201900213957)                                               430.0   
(v3_newfeat30_score<=0.048337794840335846)                                                                                 1011.0   

                                                                                                                                  \
shouxin_month                                                                                                             202009   
(score_orig<=-0.15712326765060425) & (v3_newfeat30_score<=0.02879201900213957)                                             100.0   
(score_bc1<=0.019156343303620815) & (v3_newfeat30_score>0.02879201900213957) & (v3_newfeat30_score<=0.048337794840335846)   61.0   
(v3_newfeat30_score<=0.02879201900213957)                                                                                  440.0   
(score_orig>-0.15712326765060425) & (v3_newfeat30_score<=0.02879201900213957)                                              340.0   
(v3_newfeat30_score<=0.048337794840335846)                                                                                 787.0   

                                                                                                                                   \
shouxin_month                                                                                                                 All   
(score_orig<=-0.15712326765060425) & (v3_newfeat30_score<=0.02879201900213957)                                              200.0   
(score_bc1<=0.019156343303620815) & (v3_newfeat30_score>0.02879201900213957) & (v3_newfeat30_score<=0.048337794840335846)   123.0   
(v3_newfeat30_score<=0.02879201900213957)                                                                                   970.0   
(score_orig>-0.15712326765060425) & (v3_newfeat30_score<=0.02879201900213957)                                               770.0   
(v3_newfeat30_score<=0.048337794840335846)                                                                                 1798.0   

                                                                                                                               fpd1  \
shouxin_month                                                                                                                202008   
(score_orig<=-0.15712326765060425) & (v3_newfeat30_score<=0.02879201900213957)                                             0.000000   
(score_bc1<=0.019156343303620815) & (v3_newfeat30_score>0.02879201900213957) & (v3_newfeat30_score<=0.048337794840335846)  0.000000   
(v3_newfeat30_score<=0.02879201900213957)                                                                                  0.035849   
(score_orig>-0.15712326765060425) & (v3_newfeat30_score<=0.02879201900213957)                                              0.044186   
(v3_newfeat30_score<=0.048337794840335846)                                                                                 0.062315   

                                                                                                                                     \
shouxin_month                                                                                                                202009   
(score_orig<=-0.15712326765060425) & (v3_newfeat30_score<=0.02879201900213957)                                           

In [16]:
# 生成特征重要性和高频区间
varImp = atsrRuleVarImp(df_class_reg, rule_col='rule_str')
varImp[0].style

,var_name,imp
0,score_add,46
1,score_bc1,22
2,score_bc2,10
3,score_chal,36
4,score_orig,36
5,v3_newfeat30_score,24


In [17]:
varImp[1].style

,var_name,dir,threshold,rule
0,score_add,<=,0.016727,(score_add<=0.016726970672607422)
1,score_bc1,<=,0.028787,(score_bc1<=0.02878708578646183)
2,score_bc2,<=,0.029576,(score_bc2<=0.029576282016932964)
3,score_chal,<=,0.324623,(score_chal<=0.32462286949157715)
4,score_orig,<=,1.936053,(score_orig<=1.9360529780387878)
5,v3_newfeat30_score,<=,0.048338,(v3_newfeat30_score<=0.048337794840335846)
6,score_add,>,0.016727,(score_add>0.016726970672607422)
7,score_bc1,>,0.021318,(score_bc1>0.02131775487214327)
8,score_bc2,>,0.015210,(score_bc2>0.01520993746817112)
9,score_chal,>,0.260300,(score_chal>0.2603002041578293)


### astr搜索工具特点

1、astr搜索工具提供了分类和回归两种规则生成方法，可根据不同任务目标进行选择；

2、astr搜索工具考虑维度广阔，对全部决策过程进行遍历，可以输出决策过程中的全决策路径规则以及对应的风险表现,同时还可以输出高频出现在规则中的变量清单和策略分支；

3、astr搜索工具执行高效，生成规则全面，但需用户自己根据不同的要求对工具生成的规则进行筛选形成策略。

## 策略工具结果对比

### gbl搜索工具执行结果在OOT样本上的表现

In [20]:
# 根据上表，选择策略1，在新样本上查看其效果。
final_rule = 'v3_newfeat30_score<=0.026 or score_orig<=-0.488 or score_bc1<=0.012 and not(score_chal>0.39)'

result_df = []
for y in ['1']:
    temp_df = data[(data['if_t'+y]==1)&(data.shouxin_date>='2020-10-01')]
    pass_df = temp_df.query(final_rule)
    result_df.append(pd.DataFrame((pass_df.shape[0],round(pass_df['fpd'+y].mean(),3),round(pass_df['fpd'+y].mean()/temp_df[temp_df['if_t'+y]==1]['fpd'+y].mean(),2),pass_df['fpd'+y].sum(),round(pass_df.shape[0]/temp_df[temp_df['if_t'+y]==1].shape[0],3),'fpd'+y),index=['cnt','risk','lift','bad_cnt','ar','y']).T)
result_df = pd.concat(result_df, axis=0)
result_df
#  效果尚可，同预期相近

,cnt,risk,lift,bad_cnt,ar,y
0,208,0.034,0.4,7.0,0.295,fpd1


效果尚可，同预期相近。

### frdm搜索工具执行结果在OOT样本上的表现

In [21]:
final_rule = '(score_add<=0.017251 or score_orig<=-0.175131 or v3_newfeat30_score<=0.025993 ) and not ( v3_newfeat30_score>=0.067697 )'
result_df = []
for y in ['1']:
    temp_df = data[(data['if_t'+y]==1)&(data.shouxin_date>='2020-10-01')]
    pass_df = temp_df.query(final_rule)
    result_df.append(pd.DataFrame((pass_df.shape[0],round(pass_df['fpd'+y].mean(),3),round(pass_df['fpd'+y].mean()/temp_df[temp_df['if_t'+y]==1]['fpd'+y].mean(),2),pass_df['fpd'+y].sum(),round(pass_df.shape[0]/temp_df[temp_df['if_t'+y]==1].shape[0],3),'fpd'+y),index=['cnt','risk','lift','bad_cnt','ar','y']).T)
result_df = pd.concat(result_df, axis=0)
result_df

,cnt,risk,lift,bad_cnt,ar,y
0,301,0.04,0.48,12.0,0.428,fpd1


效果达到要求，但对比gbl策略搜索工具的结果，通过率和风险均稍高。